# 14 · Foundry IQ knowledge bases

## Goal

Build a Foundry IQ knowledge base spanning multiple sources for supplier
risk signals, attach it to the agent, and run the same golden set against
it and against the Track 1 native knowledge from `03`-`05` — the comparison
is the actual point of this notebook.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from csx.config import load_settings
settings = load_settings()
settings.require("FOUNDRY_PROJECT_ENDPOINT")


## Concept

Foundry IQ knowledge bases do multi-source agentic retrieval — the KB
itself decides how to query across its sources per question, rather than a
single fixed retrieval path per source the way Track 1's native knowledge
sources do. That's a genuinely different tradeoff (more adaptive, more
opaque, a separate cost surface) and the only honest way to present it is
side by side on the same eval cases, not as a strict upgrade.


## Build


In [ ]:
# Build the KB over supplier risk sources — illustrative shape; the Foundry
# SDK's exact KB-build call is the one surface most likely to have moved
# since the 17 Aug 2026 doc pass, so check current docs before running live.
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

project = AIProjectClient(endpoint=settings.get("FOUNDRY_PROJECT_ENDPOINT"), credential=DefaultAzureCredential())

kb = project.knowledge_bases.create_or_update(
    name="supplier-risk-kb",
    sources=[
        {"type": "azure_ai_search", "index": "crd-addenda-index"},
        {"type": "web", "allowlist": ["reuters.com", "bloomberg.com"]},
    ],
)
print(kb)


### Attach to the agent


In [ ]:
import yaml
from pathlib import Path
workspace = Path("../agents/contract-renewal-desk")

source = {
    "id": "foundry-iq-supplier-risk",
    "type": "foundry_iq_knowledge_base",
    "displayName": "Supplier risk (Foundry IQ)",
    "description": "Multi-source agentic retrieval over addenda + curated financial news for supplier risk questions.",
    "knowledgeBaseId": "supplier-risk-kb",
}
(workspace / "knowledge" / "foundry-iq-supplier-risk.yaml").write_text(yaml.dump(source, sort_keys=False))

from csx.pac import copilot_push
import subprocess
copilot_push(workspace)
subprocess.run(["pac", "copilot", "publish", "--name", "crd_contract-renewal-desk"], check=True)


## Verify

Same harness, same golden set, every notebook.


Run the same knowledge cases against native (Track 1) and Foundry IQ, side by side.


In [ ]:
from csx.clients import get_copilot_client
from csx.verify import run_suite, load_golden
from csx.cost import CreditMeter

client = get_copilot_client(settings, delegated=True)
meter = CreditMeter(environment_id=settings.get("DATAVERSE_ENV_ID"))

knowledge_cases = load_golden(tags=["knowledge"])
foundry_iq_cases = load_golden(tags=["foundry-iq"])

native_suite = run_suite(client, cases=knowledge_cases, credit_meter=meter, min_pass_rate=0.75)
foundry_iq_suite = run_suite(client, cases=foundry_iq_cases, credit_meter=meter, min_pass_rate=0.75)

print(f"native Track 1 pass_rate={native_suite.pass_rate:.0%} p50={native_suite.p50_latency_ms:.0f}ms")
print(f"Foundry IQ    pass_rate={foundry_iq_suite.pass_rate:.0%} p50={foundry_iq_suite.p50_latency_ms:.0f}ms")


## Cost


In [ ]:
meter.report_cost("14", budget=settings.get("COPILOT_CREDIT_BUDGET"),
                   delta_credits=(native_suite.total_credits + foundry_iq_suite.total_credits),
                   note="KB build (separate Foundry billing) + side-by-side comparison run")


## Teardown


In [ ]:
print("No teardown — Foundry IQ KB persists for 15's serverless comparison; both sources stay attached through 25 unless removed for cost reasons.")
